## Setup
Initialize Earth Engine, geemap, and pandas.

In [ ]:
import ee
import geemap
import pandas as pd
from scipy.stats import spearmanr

ee.Initialize()

In [ ]:
Map = geemap.Map()

## Helper Functions

In [ ]:
def explode_multi_points(fc):
    """Flatten MultiPoint/GeometryCollection features into individual Point features."""
    def explode_feature(feature):
        geom = feature.geometry()
        geom_type = geom.type()
        is_container = ee.List(['MultiPoint', 'GeometryCollection']).contains(geom_type)

        def container():
            geoms = ee.List(geom.geometries())
            return ee.FeatureCollection(
                geoms.map(lambda g: ee.Feature(ee.Geometry(g)).copyProperties(feature))
            )

        def not_container():
            return ee.FeatureCollection([feature])

        return ee.Algorithms.If(is_container, container(), not_container())

    return ee.FeatureCollection(fc.map(explode_feature).flatten())


def sample_image_mean(img, geom, scale=10):
    """Return the mean of the first band of an image over a geometry at the given scale (m)."""
    band = ee.String(img.bandNames().get(0))
    mean_dict = img.reduceRegion(reducer=ee.Reducer.mean(), geometry=geom, scale=scale, maxPixels=1e8)
    return ee.Number(mean_dict.get(band))

## Data Loading
Load interpreter feature collections (certain and conf variants) and base-learner images.
Each interpreter collection is exploded to individual points, tagged with a `source` label,
then merged into a single collection per interpreter.

In [ ]:
# Load and explode interpreter feature collections
interpreters = {
    1: ('projects/ee-islamkm/assets/interpreter1_200pts_certain', 'projects/ee-islamkm/assets/interpreter1_200pts'),
    2: ('projects/ee-islamkm/assets/interpreter2_200pts_certain', 'projects/ee-islamkm/assets/interpreter2_200pts'),
    3: ('projects/ee-islamkm/assets/interpreter3_200pts_certain', 'projects/ee-islamkm/assets/interpreter3_200pts'),
}

fc1, fc2, fc3 = [
    explode_multi_points(ee.FeatureCollection(cert)).map(lambda f: f.set('source', 'cert')).merge(
        ee.FeatureCollection(conf).map(lambda f: f.set('source', 'conf'))
    )
    for cert, conf in interpreters.values()
]

# Load base-learner images
knn    = ee.Image('projects/ee-islamkm/assets/baselearner_knn_mngrv')
logreg = ee.Image('projects/ee-islamkm/assets/baselearner_logreg_mngrv')
rf     = ee.Image('projects/ee-islamkm/assets/baselearner_rf_mngrv')
svc    = ee.Image('projects/ee-islamkm/assets/baselearner_svc_mgrv')
xgb    = ee.Image('projects/ee-islamkm/assets/baselearner_xgb_mgrv')

## Per-Point Statistics
For each point in interpreter 1's collection, compute:
- Mean and std of interpreter labels within a 10 m buffer
- Mean and std of each base-learner prediction within the same buffer

In [ ]:
def compute_point_stats(feat):
    """Attach interpreter and model mean/std properties to a feature."""
    geom = feat.geometry()
    buf = geom.buffer(10)  # 10 m spatial window

    # Interpreter means from val_int within the buffer
    i1 = ee.Number(fc1.filterBounds(buf).aggregate_mean('val_int'))
    i2 = ee.Number(fc2.filterBounds(buf).aggregate_mean('val_int'))
    i3 = ee.Number(fc3.filterBounds(buf).aggregate_mean('val_int'))

    # Interpreter std via E[x^2] - (E[x])^2
    interp_mean = i1.add(i2).add(i3).divide(3)
    interp_mean_sq = i1.pow(2).add(i2.pow(2)).add(i3.pow(2)).divide(3)
    interp_std = interp_mean_sq.subtract(interp_mean.pow(2)).max(0).sqrt()

    # Base-learner means
    knn_v    = sample_image_mean(knn,    buf)
    logreg_v = sample_image_mean(logreg, buf)
    rf_v     = sample_image_mean(rf,     buf)
    svc_v    = sample_image_mean(svc,    buf)
    xgb_v    = sample_image_mean(xgb,    buf)

    # Model std across the five base-learner values
    models_mean = knn_v.add(logreg_v).add(rf_v).add(svc_v).add(xgb_v).divide(5)
    models_mean_sq = knn_v.pow(2).add(logreg_v.pow(2)).add(rf_v.pow(2)).add(svc_v.pow(2)).add(xgb_v.pow(2)).divide(5)
    models_std = models_mean_sq.subtract(models_mean.pow(2)).max(0).sqrt()

    return feat.set({
        'i1_mean': i1, 'i2_mean': i2, 'i3_mean': i3,
        'interp_std': interp_std,
        'knn_mean': knn_v, 'logreg_mean': logreg_v, 'rf_mean': rf_v,
        'svc_mean': svc_v, 'xgb_mean': xgb_v,
        'models_std': models_std
    })

## Run & Export to DataFrame

In [ ]:
# Map stats over interpreter 1 features (used as reference geometry)
result_fc = fc1.map(compute_point_stats)

In [ ]:
df = geemap.ee_to_df(result_fc)

In [ ]:
df

## Spearman Correlation
Compute the Spearman rank correlation between interpreter std and model std,
dropping any rows with null values first.

In [ ]:
cols = ['i1_mean', 'i2_mean', 'i3_mean', 'interp_std',
        'knn_mean', 'logreg_mean', 'rf_mean', 'svc_mean', 'xgb_mean', 'models_std']
df = df[cols].dropna()

rho, pvalue = spearmanr(df['interp_std'].astype(float), df['models_std'].astype(float))
print(f"Spearman correlation (interp_std vs models_std): rho = {rho:.4f}, p-value = {pvalue:.4g}")

## Visualization
Joint scatter plot of interpreter std vs. base-learner std, saved as a 300-dpi PNG.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [ ]:
sns.set_style("whitegrid")

g = sns.jointplot(
    x='interp_std',
    y='models_std',
    data=df,
    kind='scatter',
    height=6,
    marginal_kws=dict(bins=20, fill=True, color='blue', alpha=0.6),
    joint_kws=dict(color='lightgreen', edgecolor='black', linewidth=0.7)
)

g.ax_joint.set_xlabel('Interpreter stdev')
g.ax_joint.set_ylabel('Base learner stdev')
g.ax_joint.grid(True, linestyle='--', alpha=0.4)
g.fig.suptitle(' ', y=1.02, fontsize=14, fontweight='bold')

output_path = Path(r'D:\Research\uncertainty_paper\interp_vs_models_jointplot.png')
output_path.parent.mkdir(parents=True, exist_ok=True)
g.fig.savefig(output_path, dpi=300, bbox_inches='tight')

plt.show()